# Arid Ecosystem Metatranscriptomes
# Data retrieve
# September 2025

In [6]:
%%bash

# Extract SRR column to a plain list

cut -d',' -f1 SraRunTable.csv | grep ^SRR > SRR.txt
wc -l SRR.txt   # should be ~56


56 SRR.txt


# SRA-tools

## Prefetch downloads the .sra archive safely, retries if needed, and checks integrity.

In [16]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=prefetch
#SBATCH --nodes=1
#SBATCH --ntasks=94
#SBATCH --time=48:00:00   
#SBATCH --partition=standard
#SBATCH --account=tfaily
#SBATCH --mail-user=vfreirezapata@email.arizona.edu
#SBATCH --mail-type=ALL
#SBATCH -o %x-%j.out
#SBATCH --get-user-env 

conda init
source ~/.bashrc

conda activate /xdisk/tfaily/vfreirezapata/env_new/sra_tools

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"
OUTDIR="/xdisk/tfaily/vfreirezapata/metat_2025_final/raw_reads"
mkdir -p "$OUTDIR"

SRR=$1

echo "Downloading $SRR into $OUTDIR ..."
prefetch "$SRR" -O "$OUTDIR"

echo "Validating $SRR ..."
vdb-validate "$OUTDIR/${SRR}/${SRR}.sra"
    
' > scripts/prefetch.slurm

In [17]:
%%bash

while read SRR; do
    sbatch scripts/prefetch.slurm $SRR
done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 15850504
Submitted batch job 15850505
Submitted batch job 15850506
Submitted batch job 15850507
Submitted batch job 15850508
Submitted batch job 15850509
Submitted batch job 15850510
Submitted batch job 15850511
Submitted batch job 15850512
Submitted batch job 15850513
Submitted batch job 15850514
Submitted batch job 15850515
Submitted batch job 15850516
Submitted batch job 15850517
Submitted batch job 15850518
Submitted batch job 15850519
Submitted batch job 15850520
Submitted batch job 15850521
Submitted batch job 15850522
Submitted batch job 15850523
Submitted batch job 15850524
Submitted batch job 15850525
Submitted batch job 15850526
Submitted batch job 15850527
Submitted batch job 15850528
Submitted batch job 15850529
Submitted batch job 15850530
Submitted batch job 15850531
Submitted batch job 15850532
Submitted batch job 15850533
Submitted batch job 15850534
Submitted batch job 15850535
Submitted batch job 15850536
Submitted batch job 15850537
Submitted batc

## fasterq-dump converts the .sra to FASTQ.

In [1]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=fasterq
#SBATCH --nodes=1
#SBATCH --ntasks=94
#SBATCH --time=48:00:00   
#SBATCH --partition=standard
#SBATCH --account=tfaily
#SBATCH --mail-user=vfreirezapata@email.arizona.edu
#SBATCH --mail-type=ALL
#SBATCH -o %x-%j.out
#SBATCH --get-user-env 

conda init
source ~/.bashrc

conda activate /xdisk/tfaily/vfreirezapata/env_new/sra_tools

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"
INDIR="/xdisk/tfaily/vfreirezapata/metat_2025_final/raw_reads/"
OUTDIR="/xdisk/tfaily/vfreirezapata/metat_2025_final/raw_reads/fastq_reads_interleaved"


SRR=$1

fasterq-dump "$INDIR/${SRR}/${SRR}.sra" --split-spot \
    -O "$OUTDIR"

' > scripts/fasterq-dump.slurm

In [2]:
%%bash

while read SRR; do
    sbatch scripts/fasterq-dump.slurm $SRR
done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 15874165
Submitted batch job 15874166
Submitted batch job 15874167
Submitted batch job 15874168
Submitted batch job 15874169
Submitted batch job 15874170
Submitted batch job 15874171
Submitted batch job 15874172
Submitted batch job 15874173
Submitted batch job 15874174
Submitted batch job 15874175
Submitted batch job 15874176
Submitted batch job 15874177
Submitted batch job 15874178
Submitted batch job 15874179
Submitted batch job 15874180
Submitted batch job 15874181
Submitted batch job 15874182
Submitted batch job 15874183
Submitted batch job 15874184
Submitted batch job 15874185
Submitted batch job 15874186
Submitted batch job 15874187
Submitted batch job 15874188
Submitted batch job 15874189
Submitted batch job 15874190
Submitted batch job 15874191
Submitted batch job 15874192
Submitted batch job 15874193
Submitted batch job 15874194
Submitted batch job 15874195
Submitted batch job 15874197
Submitted batch job 15874198
Submitted batch job 15874199
Submitted batc

## Reformatting headers 

In [3]:
%%bash
echo '#!/bin/bash
#SBATCH --job-name=reformat
#SBATCH --nodes=1
#SBATCH --ntasks=94
#SBATCH --time=2:00:00   
#SBATCH --partition=standard
#SBATCH --account=tfaily
#SBATCH --mail-user=vfreirezapata@email.arizona.edu
#SBATCH --mail-type=ALL
#SBATCH -o %x-%j.out
#SBATCH --get-user-env 

conda init
source ~/.bashrc

conda activate /xdisk/tfaily/vfreirezapata/env_new/sra_tools

PROJECT_DIR="/xdisk/tfaily/vfreirezapata/metat_2025_final"
INDIR="/xdisk/tfaily/vfreirezapata/metat_2025_final/raw_reads/fastq_reads_interleaved"
OUTDIR="/xdisk/tfaily/vfreirezapata/metat_2025_final/raw_reads/fastq_reads_interleaved/fixed"

SRR=$1

reformat.sh fixjunk=t \
  in="$INDIR/${SRR}.fastq" \
  out="$OUTDIR/${SRR}_fixed_1.fastq" \
  out2="$OUTDIR/${SRR}_fixed_2.fastq"


conda activate /xdisk/tfaily/vfreirezapata/env_new/seqkit

cat ${OUTDIR}/${SRR}_fixed_1.fastq | seqkit replace -p "\slength.*" | seqkit replace -p ".+\s" > ${OUTDIR}/${SRR}_fixed_seqkit_1.fastq
cat ${OUTDIR}/${SRR}_fixed_2.fastq | seqkit replace -p "\slength.*" | seqkit replace -p ".+\s" > ${OUTDIR}/${SRR}_fixed_seqkit_2.fastq
    
' > scripts/reformat_sh.slurm

In [4]:
%%bash

while read SRR; do
    sbatch scripts/reformat_sh.slurm $SRR
done < /xdisk/tfaily/vfreirezapata/metat_2025_final/SRR.txt

Submitted batch job 15874674
Submitted batch job 15874675
Submitted batch job 15874676
Submitted batch job 15874677
Submitted batch job 15874678
Submitted batch job 15874679
Submitted batch job 15874680
Submitted batch job 15874681
Submitted batch job 15874682
Submitted batch job 15874683
Submitted batch job 15874684
Submitted batch job 15874685
Submitted batch job 15874686
Submitted batch job 15874687
Submitted batch job 15874688
Submitted batch job 15874689
Submitted batch job 15874690
Submitted batch job 15874691
Submitted batch job 15874692
Submitted batch job 15874693
Submitted batch job 15874694
Submitted batch job 15874695
Submitted batch job 15874696
Submitted batch job 15874697
Submitted batch job 15874698
Submitted batch job 15874699
Submitted batch job 15874700
Submitted batch job 15874701
Submitted batch job 15874702
Submitted batch job 15874703
Submitted batch job 15874704
Submitted batch job 15874705
Submitted batch job 15874706
Submitted batch job 15874707
Submitted batc